In [138]:
import numpy as np
import re
from pathlib import Path

In [217]:
class PID_group(object):
    indent = '  '
    start_indent = '    '
    
    def __init__(self, pid: str):
        self.pid = pid
        self.parameters = []
        self.init = []

    def add_parameter(self, name: str, init: str, unit: str, expr: str):
        self.parameters.append({})
        self.parameters[-1]["name"] = name
        if init not in self.init:
            self.init.append(init)
        self.parameters[-1]["unit"] = unit
        self.parameters[-1]["expr"] = process_expression(expr,self.pid)

    def format_init(self):
        unique_init = np.unique(self.init)
        formatted_inits = []
        for init in unique_init:
            formatted_inits.append('ATSH'+init)
        self.formatted_init_string = ';'.join(formatted_inits)

    def format_parameter_strings(self):
        self.parameter_strings = ''
        param_indent = self.start_indent+self.indent*3
        for parameter in self.parameters:
            self.parameter_strings += (f'{self.start_indent+self.indent*2}{{\n'
                                       f'{param_indent}"name" : "{parameter["name"]}",\n'+
                                       f'{param_indent}"expression" : "{parameter["expr"]}",\n'+
                                       f'{param_indent}"unit" : "{parameter["unit"]}" }},')
        self.parameter_strings = self.parameter_strings[:-1]

    def print_json(self):
        self.format_init()
        self.format_parameter_strings()
        pid_indent = self.start_indent+self.indent
        return (f'{self.start_indent}{{\n'+
                f'{pid_indent}"pid" : "{self.pid}",\n'+
                f'{pid_indent}"pid_init" : "{self.formatted_init_string}",\n'+
                f'{pid_indent}"parameters" : {self.parameter_strings} \n'
                f'{self.start_indent}}},\n')

In [219]:
def process_expression(expr: str, pid: str):
    if(len(pid) == 4):
        return translate(expr, 0)
    elif(len(pid) == 6):
        return translate(expr, 1)
    else:
        raise ValueError 

def translate(expr: str, offset: int):
    print(f"Init. expression: {expr}")
    # add brackets
    expr = re.sub(r"\b(?<!:)([A-Za-z]?[A-Za-z]):([A-Za-z]?[A-Za-z])",r"[\1:\2",expr)
    expr = re.sub(r"([A-Za-z]?[A-Za-z]):([A-Za-z]?[A-Za-z])(?!:)\b",r"\1:\2]",expr)
    torque_column = re.compile(r"(\b[A-Za-z]?[A-Za-z]\b)")
    cols_to_replace = torque_column.findall(expr)
    split_expression = torque_column.split(expr)
    new_expr = ''
    for part in split_expression:
        if part in cols_to_replace:
            if len(part) == 2:
                new_expr += replace_double_letters(part,part,offset)
            elif len(part) == 1:
                new_expr += replace_single_letters(part,part,offset)
        else:
            new_expr += part
    print(f"Final expression: {new_expr}\n")
    return new_expr

def replace_single_letters(letter: str, expr: str, offset: int):
    val = ord(letter.lower())-ord('a') + offset + 4
    frame_type = (val-1) // 7 
    val += frame_type
    # re here is overkill and based on a previous attempt
    return re.sub(rf"\b{letter}\b",f"B{val}",expr)
    
def replace_double_letters(letters: str, expr: str, offset: int):
    val = ord(letters.lower()[1])-ord('a')+4+offset+26
    frame_type = (val-1) // 7 
    val += frame_type
    return re.sub(rf"\b{letters}\b",f"B{val}",expr)

In [220]:
fname = '/home/trh/Packages/Hyundai-Ioniq-5-Torque-Pro-PIDs/TorqueIONIQ5AWD77kWh.csv'
pids = np.loadtxt(fname,delimiter=',',dtype='str',comments='~',skiprows=1)

/tmp/ipykernel_113506/1996144467.py:2: UserWarning: Input line 2 contained no data and will not be counted towards `max_rows=50000`.  This differs from the behaviour in NumPy <=1.22 which counted lines rather than rows.  If desired, the previous behaviour can be achieved by using `itertools.islice`.
Please see the 1.23 release notes for an example on how to do this.  If you wish to ignore this warning, use `warnings.filterwarnings`.  This warning is expected to be removed in the future and is given only once per `loadtxt` call.
  pids = np.loadtxt(fname,delimiter=',',dtype='str',comments='~',skiprows=1)


In [190]:
pid_groups = []
for line in pids:
    if("val" in line[3] or len(line[2]) == 0):
        continue
    pid = line[2].lstrip('0x') 
    pid_in_list = [group.pid == pid for group in pid_groups]
    if(any(pid_in_list)):
        group = np.array(pid_groups)[pid_in_list][0]
    else:
        group = PID_group(pid)
        pid_groups.append(group)
    group.add_parameter(line[0], line[7], line[6], line[3])
# print([group.pid for group in pid_groups])
# print([group.parameters for group in pid_groups])

Init. expression: e/50
Final expression: B9/50

Init. expression: f/50
Final expression: B10/50

Init. expression: g/50
Final expression: B11/50

Init. expression: h/50
Final expression: B12/50

Init. expression: i/50
Final expression: B13/50

Init. expression: j/50
Final expression: B14/50

Init. expression: k/50
Final expression: B15/50

Init. expression: l/50
Final expression: B16/50

Init. expression: m/50
Final expression: B17/50

Init. expression: n/50
Final expression: B18/50

Init. expression: o/50
Final expression: B19/50

Init. expression: p/50
Final expression: B20/50

Init. expression: q/50
Final expression: B21/50

Init. expression: r/50
Final expression: B22/50

Init. expression: s/50
Final expression: B23/50

Init. expression: t/50
Final expression: B24/50

Init. expression: u/50
Final expression: B25/50

Init. expression: v/50
Final expression: B26/50

Init. expression: w/50
Final expression: B27/50

Init. expression: x/50
Final expression: B28/50

Init. expression: y/5

In [191]:
json = ''
for group in pid_groups:
    json += group.print_json()

In [192]:
print(json)

    {
      "pid" : "220102",
      "pid_init" : "ATSH7E4",
      "parameters" :         {
          "name" : "000_Cell Voltage 001",
          "expression" : "B9/50",
          "unit" : "V" },        {
          "name" : "000_Cell Voltage 002",
          "expression" : "B10/50",
          "unit" : "V" },        {
          "name" : "000_Cell Voltage 003",
          "expression" : "B11/50",
          "unit" : "V" },        {
          "name" : "000_Cell Voltage 004",
          "expression" : "B12/50",
          "unit" : "V" },        {
          "name" : "000_Cell Voltage 005",
          "expression" : "B13/50",
          "unit" : "V" },        {
          "name" : "000_Cell Voltage 006",
          "expression" : "B14/50",
          "unit" : "V" },        {
          "name" : "000_Cell Voltage 007",
          "expression" : "B15/50",
          "unit" : "V" },        {
          "name" : "000_Cell Voltage 008",
          "expression" : "B16/50",
          "unit" : "V" },        {
      

In [193]:
new_fname = Path(fname).with_suffix('.json')
with open(new_fname,'w') as f:
    f.write(json)